In [0]:
# Column character limits for prod.detection.viewing_content_golden
column_limits = {
    # "tvid": 250,
    # "zipcode": 10,
    # "dma": 128,
    # "tms_episode_id": 256,
    # "tivo_episode_id": 256,
    # "tms_title": 256,
    # "tivo_title": 256,
    # "tms_airdate": 40,
    # "tivo_airdate": 40,
    # "tms_channel_callsign": 30,
    # "tivo_channel_callsign": 30,
    "tms_channel_affiliate": 40,
    "tivo_channel_affiliate": 40,
    # "is_live": 1,
    # "ip_address": 32,
    # "reported_input_source": 10,
    # "content_type": 10,
    # "tuner_content_type": 10,
    # "input_category": 16,
    # "input_device": 32,
    # "app_service": 32,
    # "tuner_tms_episode_id": 256,
    # "tuner_tivo_episode_id": 256,
    # "tuner_tms_title": 256,
    # "tuner_tivo_title": 256,
    # "tuner_tms_airdate": 40,
    # "tuner_tivo_airdate": 40,
    # "tuner_tms_channel_callsign": 30,
    # "tuner_tivo_channel_callsign": 30,
    "tuner_tms_channel_affiliate": 40,
    "tuner_tivo_channel_affiliate": 40,
    # "tuner_is_live": 1,
    # "tuner_input_category": 16,
    # "tuner_input_device": 32,
    # "tuner_app_service": 32,
    # "enableaudioacr": 1,
    # "vizio_epg_channel_id": 32,
    # "vizio_epg_program_id": 32,
    # "tms_show_genre": 256,
    # "tivo_show_genre": 256,
    # "tuner_tms_show_genre": 256,
    # "tuner_tivo_show_genre": 256,
    # "tms_epi_title": 256,
    # "tivo_epi_title": 256,
    # "tuner_tms_epi_title": 256,
    # "tuner_tivo_epi_title": 256,
    # "series_id": 256,
    # "tuner_series_id": 256,
}

In [0]:
for k, v in column_limits.items():
    print(f"{k}: {v}")

In [0]:
# Build CASE WHEN expressions: 1 if column exceeds limit, 0 otherwise
case_exprs = []
for col_name, max_lim in column_limits.items():
    case_exprs.append(
        f"CASE WHEN MAX(LENGTH(CAST({col_name} AS STRING))) > {max_lim} THEN 1 ELSE 0 END AS {col_name}"
    )

# Build violated_columns array and count expressions for the outer query
flag_columns = list(column_limits.keys())
violated_columns_expr = "filter(array(" + ", ".join([f"IF({col_name} = 1, '{col_name}', NULL)" for col_name in flag_columns]) + "), x -> x IS NOT NULL)"
violated_count_expr = " + ".join([f"`{col_name}`" for col_name in flag_columns])

In [0]:
from datetime import datetime, timedelta

# Iterate one week at a time over the last 2 years
start_date = datetime.now() - timedelta(days=730)
end_date = datetime.now()
current = start_date
week_num = 0

while current < end_date:
    week_end = min(current + timedelta(days=7), end_date)
    week_start_str = current.strftime('%Y-%m-%d')
    week_end_str = week_end.strftime('%Y-%m-%d')

    violations_query = f"""
    WITH hourly_flags AS (
        SELECT session_start_hour,
            {', '.join(case_exprs)}
        FROM prod.detection.viewing_content_golden
        WHERE session_start_hour >= '{week_start_str}'
          AND session_start_hour < '{week_end_str}'
        GROUP BY session_start_hour
    )
    SELECT
        session_start_hour,
        {', '.join([f'`{col_name}`' for col_name in flag_columns])},
        {violated_columns_expr} AS violated_columns,
        ({violated_count_expr}) AS violated_column_count
    FROM hourly_flags
    WHERE ({violated_count_expr}) > 0
    ORDER BY session_start_hour
    """

    week_df = spark.sql(violations_query)

    # First week overwrites, subsequent weeks append
    if week_num == 0:
        week_df.write.mode("overwrite").saveAsTable("dev.public.viewing_content_golden_char_violations")
    else:
        week_df.write.mode("append").insertInto("dev.public.viewing_content_golden_char_violations")

    week_num += 1
    print(f"  Processed {week_num} weeks through {week_end_str} ({datetime.now().strftime('%H:%M:%S')})")

    current = week_end

print(f"Scanned {week_num} weeks. Violations saved to dev.public.viewing_content_golden_char_violations")

In [0]:
# Summary: total hours to fix and most common violated columns
violations_df = spark.table("dev.public.viewing_content_golden_char_violations")
print(f"Total hours with violations: {violations_df.count()}")
display(violations_df)

In [0]:
from pyspark.sql.functions import explode

display(violations_df.select(explode("violated_columns").alias("violated_column")).distinct())

In [0]:
# Summary: total hours to fix and most common violated columns
violations_df = spark.table("dev.public.viewing_content_golden_char_violations_new")
print(f"Total hours with violations: {violations_df.count()}")
display(violations_df)